# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [20]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import plotly.io as pio
from getpass import getpass


pio.renderers.default = "notebook_connected"


In [21]:
os.environ['OPENAI_API_KEY'] = getpass("Enter your OPENAI_API_KEY: ")
os.environ['GOOGLE_API_KEY'] = getpass("Enter your GOOGLE_API_KEY: ")
os.environ['ANTHROPIC_API_KEY'] = getpass("Enter your ANTHROPIC_API_KEY: ")
os.environ['GROQ_API_KEY'] = getpass("Enter your GROQ_API_KEY: ")

Enter your OPENAI_API_KEY:  ········
Enter your GOOGLE_API_KEY:  ········
Enter your ANTHROPIC_API_KEY:  ········
Enter your GROQ_API_KEY:  ········


In [2]:
LITE_MODE = False

load_dotenv(override=True)
# hf_token = os.environ['HF_TOKEN']
# login(hf_token, add_to_git_credential=True)
login()

In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [7]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [8]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [11]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [12]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [13]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [14]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [15]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [16]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [17]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 4915.882, Val Loss: 12004.932


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 7893.879, Val Loss: 11063.498


In [18]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [19]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$115 $86 $22 $79 $29 $115 $68 $54 $19 $176 $238 $234 $77 $65 $53 $5 $16 $13 $35 $64 $13 $94 $1 $35 $215 $161 $166 $38 $118 $42 $69 $101 $54 $22 $39 $290 $18 $50 $112 $78 $129 $6 $18 $34 $49 $49 $35 $33 $25 $153 $31 $53 $140 $12 $61 $149 $28 $120 $15 $45 $109 $3 $23 $3 $390 $15 $44 $286 $21 $176 $7 $29 $171 $112 $23 $42 $105 $21 $25 $55 $68 $70 $36 $46 $26 $136 $115 $8 $42 $145 $29 $1 $3 $8 $42 $33 $40 $38 $39 $189 $23 $21 $15 $61 $6 $6 $65 $262 $10 $155 $27 $75 $117 $14 $10 $158 $50 $76 $22 $98 $25 $137 $85 $41 $117 $57 $15 $62 $45 $17 $41 $55 $66 $27 $72 $27 $109 $6 $30 $18 $4 $72 $45 $141 $79 $38 $31 $264 $70 $5 $17 $26 $5 $33 $32 $152 $94 $1 $48 $8 $138 $4 $12 $23 $189 $27 $50 $30 $19 $60 $36 $16 $195 $50 $8 $3 $39 $18 $30 $152 $31 $22 $89 $46 $37 $50 $85 $60 $32 $26 $69 $28 $5 $191 $29 $19 $26 $32 $9 $13 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [22]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [23]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [24]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [25]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [26]:
gpt_4__1_nano(test[0])

'$250'

In [27]:
test[0].price

219.0

In [34]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $10 $120 $50 $6 $65 $11 $870 $363 $20 $30 $9 $9 $8 $41 $24 $40 $31 $64 $26 $65 $75 $182 $254 $705 $5 $251 $60 $0 $15 $10 $50 $5 $81 $60 $26 $6 $13 $160 $35 $15 $105 $70 $5 $27 $13 $70 $52 $28 $105 $125 $10 $147 $16 $8 $49 $48 $3 $86 $2 $51 $10 $54 $30 $90 $295 $25 $74 $17 $18 $30 $1 $20 $21 $126 $1 $8 $3 $30 $3 $15 $74 $12 $10 $68 $56 $0 $29 $28 $35 $5 $20 $2 $78 $1 $7 $20 $425 $20 $3 $12 $11 $1 $32 $10 $375 $14 $51 $0 $236 $49 $8 $54 $180 $15 $0 $94 $47 $29 $511 $80 $16 $0 $10 $10 $101 $29 $89 $49 $13 $65 $5 $85 $0 $85 $10 $78 $42 $16 $25 $70 $10 $114 $68 $15 $90 $35 $18 $1 $144 $22 $20 $4 $29 $101 $41 $30 $5 $411 $17 $2 $2 $140 $7 $752 $30 $15 $5 $0 $3 $120 $8 $32 $201 $8 $57 $26 $23 $546 $15 $150 $99 $100 $13 $53 $17 $10 $2 $5 $99 $0 $11 $50 $70 $10 $30 $21 $1 

In [29]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [31]:
evaluate(claude_opus_4_5, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]

$20 $34 $15 $20 $0 $50 $54 $65 $6 $45 $214 $80 $5 $24 $49 $3 $11 $20 $20 $74 $16 $6 $40 $125 $33 $254 $196 $5 $90 $65 $10 $30 $70 $50 $25 $320 $30 $43 $34 $8 $150 $55 $10 $5 $70 $0 $5 $2 $65 $72 

In [32]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [33]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]

$10 $50 $5 $24 $5 $0 $79 $50 $12 $170 $212 $130 $4 $8 $49 $6 $31 $17 $69 $20 $16 $4 $15 $25 $52 $184 $166 $3 $121 $60 $3 $30 $139 $44 $5 $170 $10 $41 $16 $9 $134 $30 $13 $85 $20 $2 $1 $3 $80 $17 

In [35]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [36]:
evaluate(gemini_2__5_flash_lite, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $114 $5 $50 $30 $110 $64 $90 $1 $270 $363 $21 $20 $4 $29 $12 $101 $5 $140 $31 $14 $4 $10 $105 $32 $233 $145 $5 $151 $64 $10 $5 $10 $55 $35 $69 $90 $31 $64 $13 $165 $15 $0 $120 $100 $20 $7 $13 $70 $48 $27 $100 $125 $10 $77 $16 $8 $80 $18 $13 $116 $33 $46 $10 $79 $10 $50 $295 $5 $144 $7 $8 $130 $1 $20 $11 $26 $1 $8 $4 $60 $3 $5 $74 $8 $10 $168 $19 $30 $21 $3 $25 $15 $10 $0 $22 $11 $72 $95 $225 $25 $33 $7 $11 $11 $32 $15 $340 $14 $121 $5 $161 $44 $22 $64 $80 $10 $5 $64 $647 $9 $161 $65 $14 $50 $5 $15 $51 $6 $74 $204 $3 $10 $5 $135 $10 $30 $30 $2 $22 $31 $50 $5 $4 $19 $3 $20 $40 $185 $8 $1 $94 $22 $10 $6 $41 $36 $31 $10 $0 $109 $14 $18 $3 $140 $12 $1352 $10 $4 $6 $0 $3 $170 $8 $37 $101 $3 $13 $56 $18 $4 $20 $225 $24 $25 $8 $73 $12 $10 $2 $35 $19 $30 $61 $25 $120 $10 $30 $9 $1 

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [37]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [38]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$1 $74 $15 $20 $10 $90 $64 $65 $11 $19 $187 $20 $0 $9 $49 $3 $21 $15 $10 $29 $6 $4 $5 $25 $37 $203 $195 $1 $91 $64 $5 $25 $20 $55 $5 $169 $30 $36 $44 $16 $150 $40 $16 $85 $30 $0 $7 $3 $75 $98 $26 $110 $225 $0 $27 $34 $3 $80 $3 $3 $96 $48 $48 $70 $159 $9 $50 $315 $15 $24 $16 $3 $120 $0 $25 $17 $6 $1 $2 $6 $0 $4 $5 $74 $15 $25 $48 $76 $20 $21 $3 $15 $5 $5 $1 $78 $1 $77 $80 $235 $30 $17 $2 $10 $49 $132 $15 $325 $1 $119 $10 $116 $1 $58 $54 $0 $8 $0 $24 $497 $8 $91 $10 $36 $0 $5 $0 $21 $11 $59 $109 $7 $5 $0 $65 $2 $24 $10 $147 $12 $6 $249 $25 $10 $24 $8 $10 $60 $35 $8 $6 $83 $29 $60 $1 $91 $31 $36 $45 $10 $90 $17 $8 $1 $40 $2 $551 $25 $0 $0 $10 $2 $170 $10 $72 $9 $6 $27 $14 $7 $174 $15 $230 $49 $25 $3 $53 $17 $10 $8 $5 $19 $12 $31 $60 $40 $11 $0 $26 $10 